remove year built
remove price per sqdt
remove floor no > total no of floors
remove grad boost
test ridge/lasso for different alpha values
skewness and distributions in eda
log transform (fix the performance of linear models)

In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
warnings.filterwarnings('ignore')

df = pd.read_csv(r'C:\Users\vsmk3\ds_first_git\india_housing_prices.csv')
df = df.drop_duplicates()
print("Shape:", df.shape)
df.head()

Shape: (250000, 23)


,ID,State,City,Locality,Property_Type,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,...,Age_of_Property,Nearby_Schools,Nearby_Hospitals,Public_Transport_Accessibility,Parking_Space,Security,Amenities,Facing,Owner_Type,Availability_Status
0,1,Tamil Nadu,Chennai,Locality_84,Apartment,1,4740,489.76,0.10,1990,...,35,10,3,High,No,No,"Playground, Gym, Garden, Pool, Clubhouse",West,Owner,Ready_to_Move
1,2,Maharashtra,Pune,Locality_490,Independent House,3,2364,195.52,0.08,2008,...,17,8,1,Low,No,Yes,"Playground, Clubhouse, Pool, Gym, Garden",North,Builder,Under_Construction
2,3,Punjab,Ludhiana,Locality_167,Apartment,2,3642,183.79,0.05,1997,...,28,9,8,Low,Yes,No,"Clubhouse, Pool, Playground, Gym",South,Broker,Ready_to_Move
3,4,Rajasthan,Jodhpur,Locality_393,Independent House,2,2741,300.29,0.11,1991,...,34,5,7,High,Yes,Yes,"Playground, Clubhouse, Gym, Pool, Garden",North,Builder,Ready_to_Move
4,5,Rajasthan,Jaipur,Locality_466,Villa,4,4823,182.90,0.04,2002,...,23,4,9,Low,No,Yes,"Playground, Garden, Gym, Pool, Clubhouse",East,Builder,Ready_to_Move


In [ ]:
df.describe().T

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 23 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   ID                              250000 non-null  int64  
 1   State                           250000 non-null  object 
 2   City                            250000 non-null  object 
 3   Locality                        250000 non-null  object 
 4   Property_Type                   250000 non-null  object 
 5   BHK                             250000 non-null  int64  
 6   Size_in_SqFt                    250000 non-null  int64  
 7   Price_in_Lakhs                  250000 non-null  float64
 8   Price_per_SqFt                  250000 non-null  float64
 9   Year_Built                      250000 non-null  int64  
 10  Furnished_Status                250000 non-null  object 
 11  Floor_No                        250000 non-null  int64  
 12  Total_Floors    

,ID,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Floor_No,Total_Floors,Age_of_Property,Nearby_Schools,Nearby_Hospitals
count,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000
mean,125000.500000,2.999396,2749.813216,254.586854,0.130597,2006.520012,14.966800,15.503004,18.479988,5.499860,5.498016
std,72168.927986,1.415521,1300.606954,141.349921,0.130747,9.808575,8.948047,8.671618,9.808575,2.878639,2.871860
min,1.000000,1.000000,500.000000,10.000000,0.000000,1990.000000,0.000000,1.000000,2.000000,1.000000,1.000000
25%,62500.750000,2.000000,1623.000000,132.550000,0.050000,1998.000000,7.000000,8.000000,10.000000,3.000000,3.000000
50%,125000.500000,3.000000,2747.000000,253.870000,0.090000,2007.000000,15.000000,15.000000,18.000000,5.000000,5.000000
75%,187500.250000,4.000000,3874.000000,376.880000,0.160000,2015.000000,23.000000,23.000000,27.000000,8.000000,8.000000
max,250000.000000,5.000000,5000.000000,500.000000,0.990000,2023.000000,30.000000,30.000000,35.000000,10.000000,10.000000


In [ ]:
pd.set_option('display.max_colwidth', None)
# Generate categorical and string column summary
desc = df.describe(include=['O'])

# Append a row containing the list of unique entries for each column
desc.loc['unique_entries'] = df[desc.columns].apply(lambda col: col.dropna().unique().tolist())

desc.T

In [ ]:
# ── Graph 5: Correlation Heatmap ─────────────────────────────


# Drop datetime/string columns before correlation
df_numeric = df.select_dtypes(include=['float64', 'int64', 'int32', 'float32'])

print(f"Using {df_numeric.shape[1]} numeric columns for heatmap")
print("Columns:", df_numeric.columns.tolist())

plt.figure(figsize=(14, 10))
corr = df_numeric.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(
    corr,
    mask=mask,
    annot=True,          # shows values inside boxes
    fmt='.2f',           # 2 decimal places
    cmap='coolwarm',
    linewidths=0.3,
    vmin=-1, vmax=1,
    annot_kws={"size": 8}
)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('heatmap.png', dpi=150)
plt.show()

In [14]:
# ── Check Missing Values ──────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

Empty DataFrame
Columns: [Missing Count, Percentage]
Index: []


In [15]:
cat_cols = df.select_dtypes(include=['object']).columns
num_cols = df.select_dtypes(include=['float64', 'int64']).columns

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Clean column names (good practice)
df.columns = df.columns.str.strip()

# Correct target column
TARGET = 'Price_in_Lakhs'

# Features & target
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Handle missing values
X = X.fillna(0)
y = y.fillna(0)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (200000, 22) | Test: (50000, 22)


In [17]:
# ── Remove Outliers using IQR ─────────────────────────────────
train = pd.concat([X_train, y_train], axis=1)
test = pd.concat([X_test, y_test], axis=1)
Q1 = train[num_cols].quantile(0.25)
Q3 = train[num_cols].quantile(0.75)
IQR = Q3 - Q1
train = train[~((train[num_cols] < (Q1 - 1.5 * IQR)) | (train[num_cols] > (Q3 + 1.5 * IQR))).any(axis=1)]
print("Shape after outlier removal:", train.shape)

Shape after outlier removal: (183936, 23)


In [18]:
def rearrange_amenities(df):
    amenities_dummies = df['Amenities'].str.get_dummies(sep=', ')
    df = pd.concat([df, amenities_dummies], axis=1)
    df = df.drop(columns=['Amenities'])
    return df

train = rearrange_amenities(train)
test = rearrange_amenities(test)
X_train = train.drop(columns=[TARGET])
y_train = train[TARGET]
X_test = test.drop(columns=[TARGET])
y_test = test[TARGET]

print(f'Train: {train.shape} | Test: {test.shape}')

Train: (183936, 27) | Test: (50000, 27)


In [19]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

cat_cols = X_train.select_dtypes(
    include=['object', 'category']
).columns

num_cols = X_train.select_dtypes(
    include='number'
).columns

encoder = OneHotEncoder(
    handle_unknown='ignore',
    drop='first',
    sparse_output=False
)

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', encoder, cat_cols),
        ('num', 'passthrough', num_cols)
    ],
    verbose_feature_names_out=False
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

In [20]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

models = {
    "Linear Regression":      LinearRegression(),
    "Ridge Regression":       Ridge(alpha=1.0),
    "Lasso Regression":       Lasso(alpha=0.1),
    "Decision Tree":          DecisionTreeRegressor(random_state=42),
    "Random Forest":          RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting":      GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = []

for name, model in models.items():
    start = time.perf_counter()
    model.fit(X_train_encoded, y_train)
    y_pred = model.predict(X_test_encoded)

    r2  = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    end = time.perf_counter()
    
    results.append({'Model': name, 'R2 Score': round(r2, 4),
                    'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'Time': round(end - start, 2)})
    print(f"{name:25s} → R²: {r2:.4f} | MAE: {mae:.2f} | RMSE: {rmse:.2f} | Time: {end - start:.2f}s")

results_df = pd.DataFrame(results).sort_values('R2 Score', ascending=False)
print("\n", results_df)

Linear Regression         → R²: -0.0320 | MAE: 123.59 | RMSE: 143.44 | Time: 6.33s
Ridge Regression          → R²: -0.2483 | MAE: 87.37 | RMSE: 157.75 | Time: 1.25s
Lasso Regression          → R²: -0.2150 | MAE: 86.71 | RMSE: 155.63 | Time: 3.24s
Decision Tree             → R²: 0.9131 | MAE: 18.17 | RMSE: 41.61 | Time: 20.72s
Random Forest             → R²: 0.9162 | MAE: 16.14 | RMSE: 40.87 | Time: 1653.20s


KeyboardInterrupt: 

In [ ]:
# ── Graph 1: R2 Score Comparison ──────────────────────────────
plt.figure(figsize=(10, 5))
bars = plt.barh(results_df['Model'], results_df['R2 Score'],
                color=plt.cm.RdYlGn(results_df['R2 Score']))
plt.xlabel('R² Score')
plt.title('Model Comparison — R² Score', fontsize=14, fontweight='bold')
plt.axvline(x=0.8, color='red', linestyle='--', label='Target R²=0.8')
plt.legend()
plt.tight_layout()
plt.savefig('r2_comparison.png')
plt.show()

In [ ]:
# ── Graph 2: Actual vs Predicted (Best Model) ─────────────────
best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]
y_pred_best = best_model.predict(X_test)

plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred_best, alpha=0.5, color='steelblue', edgecolors='k', linewidths=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Fit')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title(f'Actual vs Predicted — {best_model_name}', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('actual_vs_predicted.png')
plt.show()

In [ ]:
# ── Graph 3: Residuals Plot ───────────────────────────────────
residuals = y_test - y_pred_best

plt.figure(figsize=(10, 4))
plt.scatter(y_pred_best, residuals, alpha=0.5, color='coral')
plt.axhline(0, color='black', linestyle='--')
plt.xlabel('Predicted Price')
plt.ylabel('Residuals')
plt.title('Residuals Plot', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('residuals.png')
plt.show()

In [ ]:
# ── Graph 4: Feature Importance (Random Forest) ───────────────
rf_model = models["Random Forest"]
feat_imp = pd.Series(rf_model.feature_importances_, index=df.drop(columns=[TARGET]).columns)
feat_imp = feat_imp.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
feat_imp.plot(kind='bar', color='teal')
plt.title('Top 15 Feature Importances — Random Forest', fontsize=13, fontweight='bold')
plt.ylabel('Importance')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('feature_importance.png')
plt.show()

In [ ]:
# ── Graph 6: Price Distribution ───────────────────────────────
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
df[TARGET].hist(bins=40, color='steelblue', edgecolor='white')
plt.title('Price Distribution')
plt.xlabel('Price')

plt.subplot(1, 2, 2)
np.log1p(df[TARGET]).hist(bins=40, color='coral', edgecolor='white')
plt.title('Log(Price) Distribution')
plt.xlabel('Log Price')

plt.suptitle('House Price Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('price_dist.png')
plt.show()

In [ ]:
print("=" * 50)
print(f"  🏆 Best Model : {results_df.iloc[0]['Model']}")
print(f"  ✅ R² Score   : {results_df.iloc[0]['R2 Score']}")
print(f"  📉 MAE        : {results_df.iloc[0]['MAE']}")
print(f"  📉 RMSE       : {results_df.iloc[0]['RMSE']}")
print("=" * 50)